# 0.15 — Novelty scores & emergence detection (genAI benchmark)

**Two-part notebook:**

| Part | Goal | When to run |
|------|------|-------------|
| **A — Score analysis** | Understand headline, lexical, and edge novelty **individually** around Oct 2022–Jan 2023 | **Run first** |
| **B — Detection** | Filter → graph → Louvain → composite ranking | After reviewing Part A; set `RUN_DETECTION = True` |

**Baseline for scoring:** headlines through **2022-09-30** (so Oct–Jan analysis months share the same pre-burst history).

**Milestones (annotation only):** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

**Prerequisites:** `genai_full_meta.parquet` + `genai_full_emb_2021_2023.npy` ([`0.10`](0.10-genai-hierarchy-drilldown.ipynb)); `genai_graph_terms.parquet` ([`0.13`](0.13-genai-graph-emergence.ipynb)).

In [13]:
import re
from collections import Counter, defaultdict
from itertools import combinations
from math import log
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from networkx.algorithms.community import louvain_communities
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tqdm.auto import tqdm

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- time windows ---
BASELINE_SCORE_END = pd.Timestamp("2022-09-30")   # history for all novelty scores
ANALYSIS_START = pd.Timestamp("2022-10-01")       # Part A focus
ANALYSIS_END = pd.Timestamp("2023-01-31")
ANALYSIS_MONTHS = ["2022-10", "2022-11", "2022-12", "2023-01"]
GENAI_FOCUS_MONTHS = ["2022-12", "2023-01"]
BASELINE_END = pd.Timestamp("2022-10-31")         # Part B graph baseline
DISCOVERY_START = pd.Timestamp("2022-11-01")      # Part B detection window
DISCOVERY_END = pd.Timestamp("2022-12-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")
RANDOM_SEED = 42

RUN_DETECTION = False  # set True after Part A review

META_CACHE = OUTPUT_DIR / "genai_full_meta.parquet"
EMB_CACHE = OUTPUT_DIR / "genai_full_emb_2021_2023.npy"
TERMS_CACHE = OUTPUT_DIR / "genai_graph_terms.parquet"

BASELINE_NN_SAMPLE = 150_000
NN_BATCH = 4096
SMOOTH_ALPHA = 1.0
TOP_N = 15

# Part B config (ignored unless RUN_DETECTION)
NOVELTY_TOP_PCT = 0.03
FREQ = "W-MON"
MIN_DOC_FREQ = 5
WEEKLY_MIN_COUNT = 2
MAX_DOC_FREQ_PCT = 0.15
MIN_WEEK_HEADLINES = 500
MIN_NOVEL_WEEK_HEADLINES = 100
MIN_PPMI = 0.0
LOUVAIN_RESOLUTION = 1.0
LOUVAIN_SEED = 42
ALPHA = 1.0
MIN_COMMUNITY_SIZE = 3
MIN_HEADLINES = 5
JACCARD_LINK = 0.25
TOP_TERMS = 12
TOP_HEADLINES = 10
TOP_RANK_PRINT = 15
W_HEADLINE = 0.5
W_EDGE = 0.3
W_WORD = 0.2

HEADLINE_SCORES_PATH = OUTPUT_DIR / "genai_novelty_headline_scores.parquet"
MONTHLY_SUMMARY_PATH = OUTPUT_DIR / "genai_novelty_monthly_summary.parquet"
TOP_TERMS_EDGES_PATH = OUTPUT_DIR / "genai_novelty_top_terms_edges.parquet"
GENAI_ADJACENT_PATH = OUTPUT_DIR / "genai_novelty_genai_adjacent.parquet"
WEEKLY_SUMMARY_PATH = OUTPUT_DIR / "genai_novelty_weekly_summary.parquet"
WEEKLY_TOP_TERMS_EDGES_PATH = OUTPUT_DIR / "genai_novelty_weekly_top_terms_edges.parquet"
WEEKLY_GENAI_ADJACENT_PATH = OUTPUT_DIR / "genai_novelty_weekly_genai_adjacent.parquet"
WEEKLY_TOP_HEADLINES_PATH = OUTPUT_DIR / "genai_novelty_weekly_top_headlines.parquet"
WEEKLY_DASHBOARD_HTML_PATH = OUTPUT_DIR / "genai_novelty_weekly_dashboard.html"
ANALYSIS_HTML_PATH = OUTPUT_DIR / "genai_novelty_analysis_dashboard.html"
MONTHLY_TOP_HEADLINES_PATH = OUTPUT_DIR / "genai_novelty_monthly_top_headlines.parquet"
ANALYSIS_WEEK_START = pd.Timestamp("2022-10-01")  # drop partial pre-Oct bucket
COMMUNITIES_PATH = OUTPUT_DIR / "genai_novelty_communities.parquet"
RANKINGS_PATH = OUTPUT_DIR / "genai_novelty_rankings.parquet"
VALIDATION_PATH = OUTPUT_DIR / "genai_novelty_validation.parquet"
COMPARE_PATH = OUTPUT_DIR / "genai_novelty_compare_methods.parquet"
TIMELINE_PATH = OUTPUT_DIR / "genai_novelty_timeline.html"

FINANCE_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock", "stocks",
    "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan", "bn",
    "march", "april", "june", "july", "august", "september", "october", "november", "december",
    "rated", "buy", "sell", "hold", "neutral", "perform", "outperform", "underperform",
    "overweight", "underweight", "equal-weight", "cut", "raise", "raised", "lowers", "upgrade",
    "downgrade", "maintains", "reiterates", "est", "eps", "adj", "sees", "expects", "forecast",
    "names", "appoints", "hires", "officer", "director", "chairman", "executive", "promotes",
    "tender", "offering", "offer", "notes", "bond", "bonds", "debt", "bills", "yield",
}
TERM_STOP = FINANCE_STOP

GENAI_T1 = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle",
]
GENAI_T2 = [
    r"\bopenai\b", r"\banthropic\b", r"chatgpt-like", r"chatgpt-style",
    r"prompt engineering", r"ai chatbot", r"ai chat bot",
]
GENAI_STANDARD = re.compile("|".join(f"(?:{p})" for p in GENAI_T1 + GENAI_T2), re.I)
GENAI_TERM_HINT = re.compile(
    r"chatgpt|openai|chatbot|generative|anthropic|\bllm\b|\bllms\b|copilot|dall-e|dalle|midjourney|stable.?diffusion|foundation.?model|gpt-\d|gpt-3|gpt-4|\bgpt\b",
    re.I,
)


def normalize_terms(value) -> list:
    """Parquet stores term lists as numpy.ndarray, not Python list."""
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple)):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return []


def add_date_marker(fig, ts, text: str = "") -> None:
    """Vertical line + label; avoids plotly add_vline bugs on datetime axes."""
    x = pd.Timestamp(ts)
    fig.add_shape(
        type="line", x0=x, x1=x, y0=0, y1=1, yref="paper",
        line=dict(dash="dash", color="gray", width=1),
    )
    if text:
        fig.add_annotation(x=x, y=1.04, yref="paper", text=text, showarrow=False, font=dict(size=10))


def week_ts(week: str) -> pd.Timestamp:
    return pd.Period(week, freq=FREQ).start_time


def analysis_weeks(df: pd.DataFrame) -> list[str]:
    weeks = sorted(df["week"].unique(), key=week_ts)
    return [w for w in weeks if ANALYSIS_WEEK_START <= week_ts(w) <= ANALYSIS_END]


print(f"Part A: score analysis {ANALYSIS_START.date()}→{ANALYSIS_END.date()} | baseline through {BASELINE_SCORE_END.date()}")
print(f"Part B detection: {'ENABLED' if RUN_DETECTION else 'DISABLED — set RUN_DETECTION=True after Part A'}")

Part A: score analysis 2022-10-01→2023-01-31 | baseline through 2022-09-30
Part B detection: DISABLED — set RUN_DETECTION=True after Part A


---
# Part A — Novelty score analysis (Oct 2022–Jan 2023)

Score three novelty measures **separately** before any clustering or detection:

1. **Headline novelty** — embedding distance from baseline headlines
2. **Lexical novelty** — `-log P(term | baseline)` for extracted terms
3. **Edge novelty** — `-log P(term pair | baseline)` for co-occurring pairs

## A1. Load data (analysis + baseline slices)

In [30]:
if not META_CACHE.exists() or not EMB_CACHE.exists():
    raise FileNotFoundError("Run 0.10 first — missing meta or embedding cache")
if not TERMS_CACHE.exists():
    raise FileNotFoundError(f"Run 0.13 first — missing {TERMS_CACHE.name}")

meta = pd.read_parquet(META_CACHE).reset_index(drop=True)
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
meta["emb_idx"] = np.arange(len(meta))

emb = np.load(EMB_CACHE, mmap_mode="r")

terms_df = pd.read_parquet(TERMS_CACHE)
terms_df["date"] = pd.to_datetime(terms_df["date"]).dt.normalize()
terms_df = terms_df.drop_duplicates(["Headline", "date"], keep="first")

news = meta.merge(terms_df[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news["terms"] = news["terms"].apply(normalize_terms)
news["row_id"] = news["emb_idx"].values
news["month"] = news["date"].dt.to_period("M").astype(str)
news["week"] = news["date"].dt.to_period(FREQ).astype(str)
news["hl_lc"] = news["Headline"].str.lower()
news["is_genai"] = news["hl_lc"].str.contains(GENAI_STANDARD, na=False)

# keep only what Part A needs: baseline through Sep + analysis Oct–Jan
news = news[news.date <= ANALYSIS_END].copy()
baseline_mask = news.date <= BASELINE_SCORE_END
analysis_mask = (news.date >= ANALYSIS_START) & (news.date <= ANALYSIS_END)
baseline_df = news.loc[baseline_mask]
analysis_df = news.loc[analysis_mask].copy()

print(f"Baseline headlines (≤{BASELINE_SCORE_END.date()}): {len(baseline_df):,}")
print(f"Analysis headlines ({ANALYSIS_START.date()}→{ANALYSIS_END.date()}): {len(analysis_df):,}")
print(f"Terms populated: {(news['terms'].map(len) > 0).sum():,} / {len(news):,}")
for m in ANALYSIS_MONTHS:
    sub = analysis_df.loc[analysis_df.month == m]
    if len(sub):
        print(f"  {m}: {len(sub):,} headlines | genAI lexicon hits: {sub.is_genai.sum()}")

Baseline headlines (≤2022-09-30): 1,806,435
Analysis headlines (2022-10-01→2023-01-31): 299,883
Terms populated: 2,106,173 / 2,106,318
  2022-10: 81,113 headlines | genAI lexicon hits: 2
  2022-11: 91,946 headlines | genAI lexicon hits: 0
  2022-12: 56,600 headlines | genAI lexicon hits: 16
  2023-01: 70,224 headlines | genAI lexicon hits: 81


## A2. Headline embedding novelty

`Novelty(h) = 1 − max cos(embed(h), embed(h'))` over baseline headlines (sampled NN index).

Scored on **analysis window only** (~230k headlines, not the full 3M corpus).

In [31]:
def stratified_baseline_sample(df: pd.DataFrame, n: int, seed: int = 42) -> np.ndarray:
    ids = df["row_id"].values
    if len(ids) <= n:
        return ids
    months = df["date"].dt.to_period("M")
    rng = np.random.default_rng(seed)
    picked = []
    for period in months.unique():
        idx = ids[months.values == period]
        k = max(1, int(round(n * len(idx) / len(ids))))
        picked.append(rng.choice(idx, size=min(k, len(idx)), replace=False))
    sample = np.unique(np.concatenate(picked))
    if len(sample) > n:
        sample = rng.choice(sample, size=n, replace=False)
    return sample


def score_headline_novelty(row_ids: np.ndarray, baseline_sample: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    baseline_emb = np.asarray(emb[baseline_sample], dtype=np.float32)
    max_sims = np.empty(len(row_ids), dtype=np.float32)
    for start in tqdm(range(0, len(row_ids), NN_BATCH), desc="headline novelty"):
        batch = row_ids[start : start + NN_BATCH]
        vecs = np.asarray(emb[batch], dtype=np.float32)
        max_sims[start : start + len(batch)] = (vecs @ baseline_emb.T).max(axis=1)
    return 1.0 - max_sims, max_sims


cached = None
if HEADLINE_SCORES_PATH.exists():
    cached = pd.read_parquet(HEADLINE_SCORES_PATH)
    cached["date"] = pd.to_datetime(cached["date"]).dt.normalize()
    analysis_df = analysis_df.drop(columns=["headline_novelty", "max_baseline_sim"], errors="ignore")
    analysis_df = analysis_df.merge(
        cached[["Headline", "date", "headline_novelty", "max_baseline_sim"]],
        on=["Headline", "date"], how="left",
    )
    print(f"Loaded {HEADLINE_SCORES_PATH.name} ({len(cached):,} cached rows)")

missing = analysis_df["headline_novelty"].isna()
if missing.any():
    baseline_sample = stratified_baseline_sample(baseline_df, BASELINE_NN_SAMPLE, RANDOM_SEED)
    print(f"NN index: {len(baseline_sample):,} baseline headlines (of {len(baseline_df):,})")
    print(f"Scoring {missing.sum():,} headlines missing from cache …")
    ids = analysis_df.loc[missing, "row_id"].values
    novelty, max_sim = score_headline_novelty(ids, baseline_sample)
    analysis_df.loc[missing, "headline_novelty"] = novelty
    analysis_df.loc[missing, "max_baseline_sim"] = max_sim
    out_cols = ["Headline", "date", "month", "week", "headline_novelty", "max_baseline_sim", "is_genai"]
    new_scores = analysis_df.loc[missing, out_cols]
    combined = pd.concat([cached, new_scores], ignore_index=True) if cached is not None else new_scores
    combined = combined.drop_duplicates(["Headline", "date"], keep="last")
    combined.to_parquet(HEADLINE_SCORES_PATH, index=False)
    print(f"Updated {HEADLINE_SCORES_PATH.name} ({len(combined):,} rows)")
elif cached is None:
    baseline_sample = stratified_baseline_sample(baseline_df, BASELINE_NN_SAMPLE, RANDOM_SEED)
    print(f"NN index: {len(baseline_sample):,} baseline headlines (of {len(baseline_df):,})")
    ids = analysis_df["row_id"].values
    novelty, max_sim = score_headline_novelty(ids, baseline_sample)
    analysis_df["headline_novelty"] = novelty
    analysis_df["max_baseline_sim"] = max_sim
    analysis_df[["Headline", "date", "month", "week", "headline_novelty", "max_baseline_sim", "is_genai"]].to_parquet(
        HEADLINE_SCORES_PATH, index=False,
    )
    print(f"Wrote {HEADLINE_SCORES_PATH.name}")

print("\nHeadline novelty — full analysis window:")
print(analysis_df["headline_novelty"].describe().round(3).to_string())
print("\nBy month:")
print(
    analysis_df.groupby("month")["headline_novelty"]
    .agg(["count", "mean", "median", lambda s: s.quantile(0.95)])
    .round(3)
    .rename(columns={"<lambda_0>": "p95"})
    .to_string()
)

Loaded genai_novelty_headline_scores.parquet (298,774 cached rows)

Headline novelty — full analysis window:
count    299883.000
mean          0.236
std           0.115
min          -0.000
25%           0.151
50%           0.248
75%           0.325
max           0.762

By month:
         count   mean  median    p95
month                               
2022-10  81113  0.231   0.243  0.402
2022-11  91946  0.228   0.234  0.404
2022-12  56600  0.254   0.268  0.412
2023-01  70224  0.239   0.251  0.408


## A3. Lexical (word) novelty

`Novelty(w) = −log P(w | baseline)` with add-α smoothing over baseline document frequencies.

In [32]:
def filter_terms_list(terms: list[str]) -> list[str]:
    out = []
    for t in terms:
        if any(tok in TERM_STOP for tok in t.split()):
            continue
        if len(t) < 3:
            continue
        out.append(t)
    return out


baseline_term_df = Counter()
baseline_n_docs = len(baseline_df)
for terms in baseline_df["terms"]:
    for t in set(filter_terms_list(terms)):
        baseline_term_df[t] += 1
V = len(baseline_term_df)


def word_novelty(term: str) -> float:
    c = baseline_term_df.get(term, 0)
    p = (c + SMOOTH_ALPHA) / (baseline_n_docs + V * SMOOTH_ALPHA)
    return -log(p)


word_novelty_cache = {t: word_novelty(t) for t in baseline_term_df}
# unseen terms get max novelty
max_word_nov = max(word_novelty_cache.values()) if word_novelty_cache else 0.0


def get_word_novelty(term: str) -> float:
    if term in word_novelty_cache:
        return word_novelty_cache[term]
    p = SMOOTH_ALPHA / (baseline_n_docs + V * SMOOTH_ALPHA)
    return -log(p)


baseline_df = baseline_df.copy()
baseline_df["terms_f"] = baseline_df["terms"].map(filter_terms_list)
analysis_df["terms_f"] = analysis_df["terms"].map(filter_terms_list)
analysis_df["mean_word_novelty"] = analysis_df["terms_f"].map(
    lambda ts: float(np.mean([get_word_novelty(t) for t in set(ts)])) if ts else np.nan
)
analysis_df["max_word_novelty"] = analysis_df["terms_f"].map(
    lambda ts: float(max(get_word_novelty(t) for t in set(ts))) if ts else np.nan
)

print(f"Baseline vocabulary: {V:,} terms | max word novelty: {max_word_nov:.2f}")
print("\nMean word novelty per headline — by month:")
print(
    analysis_df.groupby("month")[["mean_word_novelty", "max_word_novelty"]]
    .mean().round(3).to_string()
)

# monthly distinct-term novelty
month_term_rows = []
for month, grp in analysis_df.groupby("month"):
    tc = Counter()
    for terms in grp["terms_f"]:
        tc.update(set(terms))
    novs = [get_word_novelty(t) for t in tc]
    month_term_rows.append({
        "month": month, "distinct_terms": len(tc),
        "mean_term_novelty": float(np.mean(novs)) if novs else 0.0,
        "median_term_novelty": float(np.median(novs)) if novs else 0.0,
        "p95_term_novelty": float(np.quantile(novs, 0.95)) if novs else 0.0,
        "share_unseen_in_baseline": round(sum(1 for t in tc if t not in baseline_term_df) / max(len(tc), 1) * 100, 1),
    })
month_term_df = pd.DataFrame(month_term_rows)
print("\nDistinct-term lexical novelty by month:")
print(month_term_df.to_string(index=False))

Baseline vocabulary: 983,360 terms | max word novelty: 14.15

Mean word novelty per headline — by month:
         mean_word_novelty  max_word_novelty
month                                       
2022-10              8.590            12.574
2022-11              8.574            12.482
2022-12              8.806            12.941
2023-01              8.634            12.591

Distinct-term lexical novelty by month:
  month  distinct_terms  mean_term_novelty  median_term_novelty  p95_term_novelty  share_unseen_in_baseline
2022-10           95912          12.837143            13.232041         14.841479                      32.8
2022-11          101360          12.881032            13.455184         14.841479                      33.8
2022-12           80570          12.764416            13.232041         14.841479                      33.2
2023-01           85950          12.792771            13.232041         14.841479                      33.3


## A4. Edge (co-occurrence) novelty

`Novelty(a,b) = −log P(a,b | baseline)` for term pairs co-occurring in the same headline.

In [33]:
def count_pairs(rows: list[list[str]]) -> Counter:
    pc = Counter()
    for terms in rows:
        u = sorted(set(terms))
        pc.update(combinations(u, 2))
    return pc


baseline_pair_df = count_pairs(baseline_df["terms_f"].tolist())
n_pair_docs = baseline_n_docs
n_pairs_vocab = len(baseline_pair_df)


def edge_novelty(a: str, b: str) -> float:
    key = (a, b) if a < b else (b, a)
    c = baseline_pair_df.get(key, 0)
    p = (c + SMOOTH_ALPHA) / (n_pair_docs + n_pairs_vocab * SMOOTH_ALPHA)
    return -log(p)


month_edge_rows = []
top_edge_rows = []
for month, grp in analysis_df.groupby("month"):
    pc = count_pairs(grp["terms_f"].tolist())
    novs = [edge_novelty(a, b) for a, b in pc]
    month_edge_rows.append({
        "month": month,
        "distinct_pairs": len(pc),
        "mean_edge_novelty": float(np.mean(novs)) if novs else 0.0,
        "median_edge_novelty": float(np.median(novs)) if novs else 0.0,
        "p95_edge_novelty": float(np.quantile(novs, 0.95)) if novs else 0.0,
        "share_unseen_pairs": round(
            sum(1 for k in pc if k not in baseline_pair_df) / max(len(pc), 1) * 100, 1
        ),
    })
    ranked = sorted(
        [(edge_novelty(a, b), pc[(a, b)], a, b) for a, b in pc],
        reverse=True,
    )[:TOP_N]
    for rank, (nov, cnt, a, b) in enumerate(ranked, 1):
        top_edge_rows.append({
            "month": month, "rank": rank, "term_a": a, "term_b": b,
            "edge_novelty": round(nov, 3), "cooc_count": cnt,
            "unseen_in_baseline": (a, b) not in baseline_pair_df and (b, a) not in baseline_pair_df,
        })

month_edge_df = pd.DataFrame(month_edge_rows)
print("Edge novelty by month:")
print(month_edge_df.to_string(index=False))

top_edge_df = pd.DataFrame(top_edge_rows)
print(f"\nTop {TOP_N} novel edges per month (first 5 rows each):")
if top_edge_df.empty:
    print("  (none — terms may be empty; re-run A1 after normalize_terms fix)")
else:
    for month in top_edge_df["month"].unique():
        print(f"\n  {month}")
        print(top_edge_df.loc[top_edge_df.month == month].head(5).to_string(index=False))

Edge novelty by month:
  month  distinct_pairs  mean_edge_novelty  median_edge_novelty  p95_edge_novelty  share_unseen_pairs
2022-10         1111633          15.790551            16.723134         16.723134                54.1
2022-11         1189228          15.819830            16.723134         16.723134                55.1
2022-12          878171          15.808593            16.723134         16.723134                56.5
2023-01          971268          15.794569            16.723134         16.723134                55.2

Top 15 novel edges per month (first 5 rows each):

  2022-10
  month  rank      term_a      term_b  edge_novelty  cooc_count  unseen_in_baseline
2022-10     1      strike    transnet        16.723          26                True
2022-10     2       naver    poshmark        16.723          23                True
2022-10     3 chile close       close        16.723          20                True
2022-10     4       chile chile close        16.723          20      

## A5. Combined monthly summary + top terms

In [34]:
monthly_summary = []
top_term_rows = []

for month, grp in analysis_df.groupby("month"):
    tc = Counter()
    for terms in grp["terms_f"]:
        tc.update(set(terms))
    mt = month_term_df.loc[month_term_df.month == month].iloc[0]
    me = month_edge_df.loc[month_edge_df.month == month].iloc[0]
    monthly_summary.append({
        "month": month,
        "n_headlines": len(grp),
        "genai_lexicon_hits": int(grp.is_genai.sum()),
        "genai_share_bp": round(grp.is_genai.mean() * 10_000, 2),
        "headline_novelty_mean": round(grp.headline_novelty.mean(), 3),
        "headline_novelty_median": round(grp.headline_novelty.median(), 3),
        "headline_novelty_p95": round(grp.headline_novelty.quantile(0.95), 3),
        "mean_word_novelty_per_hl": round(grp.mean_word_novelty.mean(), 3),
        "mean_term_novelty": mt["mean_term_novelty"],
        "share_unseen_terms_pct": mt["share_unseen_in_baseline"],
        "mean_edge_novelty": me["mean_edge_novelty"],
        "share_unseen_pairs_pct": me["share_unseen_pairs"],
    })
    ranked_terms = sorted(
        [(get_word_novelty(t), tc[t], t) for t in tc], key=lambda x: (-x[0], -x[1]),
    )[:TOP_N]
    for rank, (nov, cnt, term) in enumerate(ranked_terms, 1):
        top_term_rows.append({
            "month": month, "rank": rank, "term": term,
            "word_novelty": round(nov, 3), "doc_freq": cnt,
            "unseen_in_baseline": term not in baseline_term_df,
            "genai_hint": bool(GENAI_TERM_HINT.search(term)),
        })

monthly_summary_df = pd.DataFrame(monthly_summary)
monthly_summary_df.to_parquet(MONTHLY_SUMMARY_PATH, index=False)
top_term_df = pd.DataFrame(top_term_rows)
top_edges_out = top_edge_df.copy()
pd.concat([
    top_term_df.assign(kind="term"),
    top_edges_out.assign(kind="edge"),
], ignore_index=True).to_parquet(TOP_TERMS_EDGES_PATH, index=False)

print("=== Monthly novelty summary (Oct 2022–Jan 2023) ===")
print(monthly_summary_df.to_string(index=False))
print(f"\nWrote {MONTHLY_SUMMARY_PATH.name} + {TOP_TERMS_EDGES_PATH.name}")

monthly_hl_rows = []
TOP_HL_PER_MONTH = 5
for month, grp in analysis_df.groupby("month"):
    for kind, sub in [("all", grp), ("genai", grp.loc[grp.is_genai])]:
        if sub.empty:
            continue
        top = sub.nlargest(TOP_HL_PER_MONTH, "headline_novelty")
        for rank, (_, r) in enumerate(top.iterrows(), 1):
            monthly_hl_rows.append({
                "month": month, "kind": kind, "rank": rank,
                "date": r.date, "headline_novelty": round(r.headline_novelty, 3),
                "is_genai": bool(r.is_genai), "Headline": r.Headline,
            })
pd.DataFrame(monthly_hl_rows).to_parquet(MONTHLY_TOP_HEADLINES_PATH, index=False)
print(f"Wrote {MONTHLY_TOP_HEADLINES_PATH.name}")



print("\nTop novel terms with genAI hint:")
genai_terms = top_term_df.loc[top_term_df.genai_hint]
print(genai_terms.to_string(index=False) if len(genai_terms) else "  (none in top-15/month)")

print("\nTop headline-novelty headlines per month (genAI flagged):")
for month, grp in analysis_df.groupby("month"):
    top = grp.nlargest(5, "headline_novelty")[["date", "headline_novelty", "is_genai", "Headline"]]
    print(f"\n  {month}")
    for _, r in top.iterrows():
        flag = " [genAI]" if r.is_genai else ""
        print(f"    {r.date.date()} nov={r.headline_novelty:.3f}{flag}  {r.Headline[:90]}")

=== Monthly novelty summary (Oct 2022–Jan 2023) ===
  month  n_headlines  genai_lexicon_hits  genai_share_bp  headline_novelty_mean  headline_novelty_median  headline_novelty_p95  mean_word_novelty_per_hl  mean_term_novelty  share_unseen_terms_pct  mean_edge_novelty  share_unseen_pairs_pct
2022-10        81113                   2            0.25                  0.231                    0.243                 0.402                     8.590          12.837143                    32.8          15.790551                    54.1
2022-11        91946                   0            0.00                  0.228                    0.234                 0.404                     8.574          12.881032                    33.8          15.819830                    55.1
2022-12        56600                  16            2.83                  0.254                    0.268                 0.412                     8.806          12.764416                    33.2          15.808593                 

NameError: name 'MONTHLY_TOP_HEADLINES_PATH' is not defined

## A5b. GenAI-adjacent terms & pairs (Dec 2022 – Jan 2023)

All terms/pairs matching `GENAI_TERM_HINT` in the two focus months, ranked by lexical/edge novelty (no frequency cutoff).

In [ ]:
def is_genai_adjacent_term(term: str) -> bool:
    return bool(GENAI_TERM_HINT.search(term))


def is_genai_adjacent_pair(a: str, b: str) -> bool:
    return is_genai_adjacent_term(a) or is_genai_adjacent_term(b)


    genai_adj_rows = []
for month in GENAI_FOCUS_MONTHS:
    grp = analysis_df.loc[analysis_df.month == month]
    if grp.empty:
        continue
    tc = Counter()
    for terms in grp["terms_f"]:
        tc.update(set(terms))
    genai_terms = sorted(
        [(get_word_novelty(t), tc[t], t) for t in tc if is_genai_adjacent_term(t)],
        key=lambda x: (-x[0], -x[1]),
    )
    for rank, (nov, cnt, term) in enumerate(genai_terms, 1):
        genai_adj_rows.append({
            "month": month, "kind": "term", "rank": rank, "term": term,
            "word_novelty": round(nov, 3), "doc_freq": cnt,
            "unseen_in_baseline": term not in baseline_term_df,
        })
    pc = count_pairs(grp["terms_f"].tolist())
    genai_pairs = sorted(
        [(edge_novelty(a, b), pc[(a, b)], a, b) for a, b in pc if is_genai_adjacent_pair(a, b)],
        key=lambda x: (-x[0], -x[1]),
    )
    for rank, (nov, cnt, a, b) in enumerate(genai_pairs, 1):
        genai_adj_rows.append({
            "month": month, "kind": "edge", "rank": rank,
            "term_a": a, "term_b": b,
            "edge_novelty": round(nov, 3), "cooc_count": cnt,
            "unseen_in_baseline": (a, b) not in baseline_pair_df and (b, a) not in baseline_pair_df,
        })

genai_adj_df = pd.DataFrame(genai_adj_rows)
genai_adj_df.to_parquet(GENAI_ADJACENT_PATH, index=False)
print(f"Wrote {GENAI_ADJACENT_PATH.name} ({len(genai_adj_df):,} rows)")

for month in GENAI_FOCUS_MONTHS:
    print(f"\n=== GenAI-adjacent terms — {month} ===")
    terms = genai_adj_df.loc[(genai_adj_df.month == month) & (genai_adj_df.kind == "term")].sort_values("rank")
    if terms.empty:
        print("  (none)")
    else:
        print(terms[["rank", "term", "word_novelty", "doc_freq", "unseen_in_baseline"]].head(TOP_N).to_string(index=False))

    print(f"\n=== GenAI-adjacent pairs — {month} ===")
    edges = genai_adj_df.loc[(genai_adj_df.month == month) & (genai_adj_df.kind == "edge")].sort_values("rank")
    if edges.empty:
        print("  (none)")
    else:
        print(edges[["rank", "term_a", "term_b", "edge_novelty", "cooc_count", "unseen_in_baseline"]].head(TOP_N).to_string(index=False))

Wrote genai_novelty_genai_adjacent.parquet (848 rows)

=== GenAI-adjacent terms — 2022-12 ===
                    term  word_novelty  doc_freq  unseen_in_baseline
                 chatgpt        14.841       7.0                True
                chatbots        14.841       2.0                True
           chatgpt holds        14.841       2.0                True
             chatbot lit        14.841       1.0                True
          openai chatbot        14.841       1.0                True
microsoft-backed chatgpt        14.841       1.0                True
           chatgpt owner        14.841       1.0                True
         openai projects        14.841       1.0                True
 microsoft-backed openai        14.841       1.0                True
       gpt infraprojects        14.148       2.0               False
              gpt starts        14.148       1.0               False
                  openai        12.896       4.0               False
         

## A5c. Weekly rankings (Oct 2022–Jan 2023)

Same metrics as A4–A5b at **ISO week** (`W-MON`). Weeks with start ≥ 2022-10-01.

Exports ranked lists (`rank` column) for terms, edges, genAI-adjacent entities, and headlines.

In [ ]:
weeks = analysis_weeks(analysis_df)
weekly_summary_rows = []
weekly_term_rows = []
weekly_edge_rows = []
weekly_genai_rows = []
weekly_hl_rows = []
TOP_HL_PER_WEEK = 5

for week in weeks:
    grp = analysis_df.loc[analysis_df.week == week]
    tc = Counter()
    for terms in grp["terms_f"]:
        tc.update(set(terms))
    pc = count_pairs(grp["terms_f"].tolist())
    term_novs = [get_word_novelty(t) for t in tc]
    edge_novs = [edge_novelty(a, b) for a, b in pc]
    n_genai_adj_pairs = sum(1 for a, b in pc if is_genai_adjacent_pair(a, b))
    weekly_summary_rows.append({
        "week": week,
        "week_start": week_ts(week),
        "n_headlines": len(grp),
        "genai_lexicon_hits": int(grp.is_genai.sum()),
        "genai_share_bp": round(grp.is_genai.mean() * 10_000, 2),
        "headline_novelty_mean": round(grp.headline_novelty.mean(), 3),
        "headline_novelty_median": round(grp.headline_novelty.median(), 3),
        "headline_novelty_p95": round(grp.headline_novelty.quantile(0.95), 3),
        "mean_word_novelty_per_hl": round(grp.mean_word_novelty.mean(), 3),
        "mean_term_novelty": round(float(np.mean(term_novs)), 3) if term_novs else 0.0,
        "share_unseen_terms_pct": round(
            sum(1 for t in tc if t not in baseline_term_df) / max(len(tc), 1) * 100, 1
        ),
        "mean_edge_novelty": round(float(np.mean(edge_novs)), 3) if edge_novs else 0.0,
        "share_unseen_pairs_pct": round(
            sum(1 for k in pc if k not in baseline_pair_df) / max(len(pc), 1) * 100, 1
        ),
        "genai_adj_terms": sum(1 for t in tc if is_genai_adjacent_term(t)),
        "genai_adj_pairs": n_genai_adj_pairs,
    })
    ranked_terms = sorted(
        [(get_word_novelty(t), tc[t], t) for t in tc], key=lambda x: (-x[0], -x[1]),
    )[:TOP_N]
    for rank, (nov, cnt, term) in enumerate(ranked_terms, 1):
        weekly_term_rows.append({
            "week": week, "week_start": week_ts(week), "rank": rank, "term": term,
            "word_novelty": round(nov, 3), "doc_freq": cnt,
            "unseen_in_baseline": term not in baseline_term_df,
            "genai_hint": is_genai_adjacent_term(term),
        })
    ranked_edges = sorted(
        [(edge_novelty(a, b), pc[(a, b)], a, b) for a, b in pc], reverse=True,
    )[:TOP_N]
    for rank, (nov, cnt, a, b) in enumerate(ranked_edges, 1):
        weekly_edge_rows.append({
            "week": week, "week_start": week_ts(week), "rank": rank,
            "term_a": a, "term_b": b, "edge_novelty": round(nov, 3), "cooc_count": cnt,
            "unseen_in_baseline": (a, b) not in baseline_pair_df and (b, a) not in baseline_pair_df,
            "genai_hint": is_genai_adjacent_pair(a, b),
        })
    genai_terms = sorted(
        [(get_word_novelty(t), tc[t], t) for t in tc if is_genai_adjacent_term(t)],
        key=lambda x: (-x[0], -x[1]),
    )
    for rank, (nov, cnt, term) in enumerate(genai_terms, 1):
        weekly_genai_rows.append({
            "week": week, "week_start": week_ts(week), "kind": "term", "rank": rank,
            "term": term, "word_novelty": round(nov, 3), "doc_freq": cnt,
            "unseen_in_baseline": term not in baseline_term_df,
        })
    genai_pairs = sorted(
        [(edge_novelty(a, b), pc[(a, b)], a, b) for a, b in pc if is_genai_adjacent_pair(a, b)],
        key=lambda x: (-x[0], -x[1]),
    )
    for rank, (nov, cnt, a, b) in enumerate(genai_pairs, 1):
        weekly_genai_rows.append({
            "week": week, "week_start": week_ts(week), "kind": "edge", "rank": rank,
            "term_a": a, "term_b": b, "edge_novelty": round(nov, 3), "cooc_count": cnt,
            "unseen_in_baseline": (a, b) not in baseline_pair_df and (b, a) not in baseline_pair_df,
        })
    for kind, sub in [("all", grp), ("genai", grp.loc[grp.is_genai])]:
        if sub.empty:
            continue
        top = sub.nlargest(TOP_HL_PER_WEEK, "headline_novelty")
        for rank, (_, r) in enumerate(top.iterrows(), 1):
            weekly_hl_rows.append({
                "week": week, "week_start": week_ts(week), "kind": kind, "rank": rank,
                "date": r.date, "headline_novelty": round(r.headline_novelty, 3),
                "is_genai": bool(r.is_genai), "Headline": r.Headline,
            })

weekly_summary_df = pd.DataFrame(weekly_summary_rows)
weekly_summary_df.to_parquet(WEEKLY_SUMMARY_PATH, index=False)
weekly_top_terms_df = pd.DataFrame(weekly_term_rows)
weekly_top_edges_df = pd.DataFrame(weekly_edge_rows)
pd.concat([
    weekly_top_terms_df.assign(kind="term"),
    weekly_top_edges_df.assign(kind="edge"),
], ignore_index=True).to_parquet(WEEKLY_TOP_TERMS_EDGES_PATH, index=False)
weekly_genai_adj_df = pd.DataFrame(weekly_genai_rows)
weekly_genai_adj_df.to_parquet(WEEKLY_GENAI_ADJACENT_PATH, index=False)
weekly_top_hl_df = pd.DataFrame(weekly_hl_rows)
weekly_top_hl_df.to_parquet(WEEKLY_TOP_HEADLINES_PATH, index=False)

print(f"Weeks: {len(weeks)} ({weeks[0]} … {weeks[-1]})")
print(f"Wrote {WEEKLY_SUMMARY_PATH.name} ({len(weekly_summary_df)} rows)")
print(f"Wrote {WEEKLY_TOP_TERMS_EDGES_PATH.name} ({len(weekly_top_terms_df) + len(weekly_top_edges_df)} rows)")
print(f"Wrote {WEEKLY_GENAI_ADJACENT_PATH.name} ({len(weekly_genai_adj_df)} rows)")
print(f"Wrote {WEEKLY_TOP_HEADLINES_PATH.name} ({len(weekly_top_hl_df)} rows)")
print("\nWeekly summary (compact):")
print(weekly_summary_df[[
    "week_start", "n_headlines", "genai_lexicon_hits", "genai_share_bp",
    "headline_novelty_mean", "genai_adj_terms", "genai_adj_pairs",
]].to_string(index=False))


## A5d. Weekly dashboard — heatmap + timeline

Interactive HTML saved to `output/`; chart renders inline below.

In [ ]:
ws = weekly_summary_df.sort_values("week_start").copy()
ws["week_label"] = ws["week_start"].dt.strftime("%Y-%m-%d")

fig_hm = go.Figure(data=go.Heatmap(
    z=ws[[
        "genai_share_bp", "headline_novelty_mean", "mean_term_novelty",
        "mean_edge_novelty", "genai_adj_pairs",
    ]].T.values,
    x=ws["week_label"],
    y=[
        "genAI share (bp)", "headline nov mean", "term nov mean",
        "edge nov mean", "genAI-adj pairs",
    ],
    colorscale="Viridis",
    colorbar=dict(title="value"),
))
fig_hm.update_layout(
    title="Weekly novelty heatmap — Oct 2022–Jan 2023",
    height=320, xaxis_title="week start (Mon)",
)
fig_hm.show()

fig_w = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Headline novelty (mean)", "GenAI lexicon hits", "GenAI-adjacent pairs"),
)
fig_w.add_trace(go.Scatter(
    x=ws["week_start"], y=ws["headline_novelty_mean"],
    mode="lines+markers", name="hl mean",
), row=1, col=1)
fig_w.add_trace(go.Bar(
    x=ws["week_start"], y=ws["genai_lexicon_hits"], name="genAI hits",
), row=2, col=1)
fig_w.add_trace(go.Bar(
    x=ws["week_start"], y=ws["genai_adj_pairs"], name="genAI-adj pairs",
), row=3, col=1)
add_date_marker(fig_w, CHATGPT_LAUNCH, "ChatGPT")
fig_w.update_layout(
    title="Weekly novelty timeline — Oct 2022–Jan 2023", height=700, showlegend=False,
)
fig_w.write_html(WEEKLY_DASHBOARD_HTML_PATH)
print(f"Weekly dashboard → {WEEKLY_DASHBOARD_HTML_PATH.name}")
fig_w.show()


## A5e. Week picker — drill into one week

Set `PICK_WEEK` to any week string from the summary table (e.g. `2023-01-17/2023-01-23`).

In [ ]:
PICK_WEEK = weeks[-1]

def show_week(week: str) -> None:
    row = weekly_summary_df.loc[weekly_summary_df.week == week]
    if row.empty:
        print(f"Unknown week: {week}")
        print("Available:", weekly_summary_df["week"].tolist())
        return
    print(f"=== {week} (start {week_ts(week).date()}) ===")
    print(row.T.to_string())
    te = pd.read_parquet(WEEKLY_TOP_TERMS_EDGES_PATH)
    ga = pd.read_parquet(WEEKLY_GENAI_ADJACENT_PATH)
    hl = pd.read_parquet(WEEKLY_TOP_HEADLINES_PATH)
    print(f"\nTop {TOP_N} terms (word novelty rank):")
    print(te.loc[(te.week == week) & (te.kind == "term")][
        ["rank", "term", "word_novelty", "doc_freq", "genai_hint"]
    ].to_string(index=False))
    print(f"\nTop {TOP_N} edges (edge novelty rank):")
    print(te.loc[(te.week == week) & (te.kind == "edge")][
        ["rank", "term_a", "term_b", "edge_novelty", "cooc_count", "genai_hint"]
    ].to_string(index=False))
    gterms = ga.loc[(ga.week == week) & (ga.kind == "term")].sort_values("rank")
    print(f"\nGenAI-adjacent terms ({len(gterms)}):")
    print(gterms[["rank", "term", "word_novelty", "doc_freq"]].head(TOP_N).to_string(index=False)
        if len(gterms) else "  (none)")
    gedges = ga.loc[(ga.week == week) & (ga.kind == "edge")].sort_values("rank")
    print(f"\nGenAI-adjacent pairs ({len(gedges)}):")
    print(gedges[["rank", "term_a", "term_b", "edge_novelty", "cooc_count"]].head(TOP_N).to_string(index=False)
        if len(gedges) else "  (none)")
    print("\nTop headlines by headline-novelty rank (all):")
    for _, r in hl.loc[(hl.week == week) & (hl.kind == "all")].sort_values("rank").iterrows():
        flag = " [genAI]" if r.is_genai else ""
        print(f"  #{r.rank} {r.date.date()} {r.headline_novelty:.3f}{flag}  {r.Headline[:90]}")
    print("\nTop headlines by headline-novelty rank (genAI lexicon):")
    ghl = hl.loc[(hl.week == week) & (hl.kind == "genai")].sort_values("rank")
    if ghl.empty:
        print("  (none)")
    else:
        for _, r in ghl.iterrows():
            print(f"  #{r.rank} {r.date.date()} {r.headline_novelty:.3f}  {r.Headline[:90]}")

show_week(PICK_WEEK)


## A6. Visualizations — three scores over Oct 2022–Jan 2023

Charts render **inline** below; HTML copies are also saved to `output/`.

In [20]:
months_order = ANALYSIS_MONTHS
ms = monthly_summary_df.set_index("month").reindex(months_order)
num_cols = [
    "headline_novelty_mean", "headline_novelty_p95", "mean_term_novelty",
    "share_unseen_terms_pct", "mean_edge_novelty", "share_unseen_pairs_pct", "genai_lexicon_hits",
]
ms[num_cols] = ms[num_cols].apply(pd.to_numeric, errors="coerce")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Headline embedding novelty (mean / p95)",
        "Lexical novelty (distinct terms)",
        "Edge novelty (distinct pairs)",
        "GenAI lexicon hits (post-hoc)",
    ),
)
fig.add_trace(go.Bar(x=ms.index, y=ms["headline_novelty_mean"], name="hl mean"), row=1, col=1)
fig.add_trace(go.Bar(x=ms.index, y=ms["headline_novelty_p95"], name="hl p95"), row=1, col=1)
fig.add_trace(go.Bar(x=ms.index, y=ms["mean_term_novelty"], name="term mean"), row=1, col=2)
fig.add_trace(go.Bar(x=ms.index, y=ms["share_unseen_terms_pct"], name="% unseen terms"), row=2, col=1)
fig.add_trace(go.Bar(x=ms.index, y=ms["mean_edge_novelty"], name="edge mean"), row=2, col=1)
fig.add_trace(go.Bar(x=ms.index, y=ms["share_unseen_pairs_pct"], name="% unseen pairs"), row=2, col=2)
fig.add_trace(go.Bar(x=ms.index, y=ms["genai_lexicon_hits"], name="genAI hits"), row=2, col=2)
fig.update_layout(
    title="Novelty scores by month — Oct 2022–Jan 2023 (baseline through Sep 2022)",
    height=700, barmode="group", showlegend=True,
)
fig.write_html(ANALYSIS_HTML_PATH)
print(f"Monthly dashboard → {ANALYSIS_HTML_PATH.name}")
fig.show()

# weekly headline novelty Oct–Jan
weekly_hl = (
    analysis_df.groupby("week")["headline_novelty"]
    .agg(["mean", "median", "count"]).reset_index()
)
weekly_hl["week_ts"] = weekly_hl["week"].map(lambda w: pd.Period(w, freq=FREQ).start_time)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=weekly_hl["week_ts"], y=weekly_hl["mean"], mode="lines+markers", name="mean headline novelty",
))
fig2.add_trace(go.Scatter(
    x=weekly_hl["week_ts"], y=weekly_hl["median"], mode="lines", name="median", line=dict(dash="dot"),
))
add_date_marker(fig2, CHATGPT_LAUNCH, "ChatGPT")
fig2.update_layout(title="Weekly headline novelty — Oct 2022–Jan 2023", height=400, xaxis_title="week")
weekly_path = OUTPUT_DIR / "genai_novelty_weekly_headline.html"
fig2.write_html(weekly_path)
print(f"Weekly headline novelty → {weekly_path.name}")
fig2.show()

# distribution overlay by month
fig3 = go.Figure()
for month in ANALYSIS_MONTHS:
    sub = analysis_df.loc[analysis_df.month == month, "headline_novelty"]
    if sub.empty:
        continue
    label = pd.Period(month, freq="M").strftime("%b %Y")
    fig3.add_trace(go.Histogram(x=sub, name=label, opacity=0.5, nbinsx=40))
fig3.update_layout(title="Headline novelty distribution by month", barmode="overlay", height=360)
dist_path = OUTPUT_DIR / "genai_novelty_headline_dist.html"
fig3.write_html(dist_path)
print(f"Headline novelty distribution → {dist_path.name}")
fig3.show()

Monthly dashboard → genai_novelty_analysis_dashboard.html


Weekly headline novelty → genai_novelty_weekly_headline.html


Headline novelty distribution → genai_novelty_headline_dist.html


## A7. Part A interpretation

**Read the monthly summary table above before enabling detection.**

Questions to answer from Part A:

1. Does **headline novelty** shift between Oct → Nov → Dec (weekly plot)?
2. Do **lexical** or **edge** novelty rise in Dec/Jan when genAI lexicon hits ramp (`0.11`)?
3. Do genAI-adjacent terms/pairs (A5b / weekly A5c) show up before the Jan 2023 Microsoft–OpenAI wave?
4. Use **A5e week picker** to inspect any single week’s ranked terms, edges, and headlines.
5. Which score type looks most promising as a **pre-filter** for Part B?

When satisfied, set `RUN_DETECTION = True` in §0 and run **Part B** below.

---
# Part B — Detection pipeline (Nov–Dec 2022)

Requires Part A headline scores. Uses top `NOVELTY_TOP_PCT` novel headlines per week → PPMI graph → Louvain → composite `Novelty × Coherence × Growth`.

**Skip unless `RUN_DETECTION = True`.**

In [25]:
if not RUN_DETECTION:
    print("Part B skipped — set RUN_DETECTION = True in §0 after reviewing Part A.")
else:
    # rebuild full news through 2023 for graph baseline stats (Part B scope)
    meta_full = pd.read_parquet(META_CACHE).reset_index(drop=True)
    meta_full["date"] = pd.to_datetime(meta_full["date"]).dt.normalize()
    meta_full["emb_idx"] = np.arange(len(meta_full))
    terms_full = pd.read_parquet(TERMS_CACHE)
    terms_full["date"] = pd.to_datetime(terms_full["date"]).dt.normalize()
    terms_full = terms_full.drop_duplicates(["Headline", "date"], keep="first")
    news = meta_full.merge(terms_full[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
    news["terms"] = news["terms"].apply(normalize_terms)
    news["row_id"] = news["emb_idx"].values
    news["week"] = news["date"].dt.to_period(FREQ).astype(str)
    news["hl_lc"] = news["Headline"].str.lower()

    if HEADLINE_SCORES_PATH.exists():
        hs = pd.read_parquet(HEADLINE_SCORES_PATH)
        hs["date"] = pd.to_datetime(hs["date"]).dt.normalize()
        news = news.merge(
            hs[["Headline", "date", "headline_novelty", "max_baseline_sim"]],
            on=["Headline", "date"], how="left",
        )
        news["headline_novelty"] = news["headline_novelty"].fillna(0.0)
    else:
        raise FileNotFoundError("Run Part A first — missing headline scores cache")

    news["is_novel"] = False
    disc_mask = (news.date >= DISCOVERY_START) & (news.date <= DISCOVERY_END)
    for week, grp in news.loc[disc_mask].groupby("week"):
        thresh = grp["headline_novelty"].quantile(1 - NOVELTY_TOP_PCT)
        news.loc[grp.index[grp.headline_novelty >= thresh], "is_novel"] = True
    print(f"Part B loaded | novel headlines in discovery: {news.loc[disc_mask, 'is_novel'].sum():,}")

Part B skipped — set RUN_DETECTION = True in §0 after reviewing Part A.


In [26]:
if not RUN_DETECTION:
    pass
else:
    def build_vocab(doc_terms, n_docs):
        dfreq = Counter()
        for terms in doc_terms:
            for t in set(terms):
                dfreq[t] += 1
        max_df = int(MAX_DOC_FREQ_PCT * n_docs)
        return {t for t, c in dfreq.items() if MIN_DOC_FREQ <= c <= max_df}

    def filter_terms_v(terms, vocab):
        return [t for t in terms if t in vocab and not any(tok in TERM_STOP for tok in t.split())]

    def build_ppmi_graph(rows, min_count=WEEKLY_MIN_COUNT):
        tc, pc = Counter(), Counter()
        n = len(rows)
        if n == 0:
            return nx.Graph()
        for phrases in rows:
            u = set(phrases)
            tc.update(u)
            pc.update(combinations(sorted(u), 2))
        g = nx.Graph()
        for term, c in tc.items():
            if c >= min_count:
                g.add_node(term, count=c)
        for (a, b), c in pc.items():
            if a not in g or b not in g:
                continue
            p_ab, p_a, p_b = c / n, tc[a] / n, tc[b] / n
            ppmi = max(0.0, log(p_ab / (p_a * p_b)) if p_a * p_b > 0 else 0.0)
            if ppmi > MIN_PPMI:
                g.add_edge(a, b, weight=ppmi, count=c)
        return g

    def wdeg(g, n):
        return sum(g[n][x].get("weight", 0) for x in g.neighbors(n))

    def coherence(g, terms):
        ws = [g[a][b].get("weight", 0) if g.has_edge(a, b) else 0 for a, b in combinations(sorted(terms), 2)]
        return float(np.mean(ws)) if ws else 0.0

    def detect_comms(g):
        if g.number_of_nodes() == 0:
            return []
        out = []
        for idx, ts in enumerate(louvain_communities(g, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=LOUVAIN_SEED)):
            if len(ts) < MIN_COMMUNITY_SIZE:
                continue
            terms = set(ts)
            edges = [(a, b, g[a][b].get("count", 0)) for a, b in combinations(sorted(terms), 2) if g.has_edge(a, b)]
            out.append({"community_id": idx, "terms": terms, "internal_edges": edges, "coherence": coherence(g, terms)})
        return out

    def week_ts(w):
        return pd.Period(w, freq=FREQ).start_time

    def jaccard(a, b):
        return len(a & b) / len(a | b) if (a or b) else 0.0

    def ek(a, b):
        return (a, b) if a < b else (b, a)

    vocab = build_vocab(news["terms"].map(filter_terms_list), len(news))
    news["terms_f"] = news["terms"].map(lambda ts: filter_terms_v(filter_terms_list(ts), vocab))
    week_counts = news.groupby("week").size().to_dict()
    twc = defaultdict(lambda: defaultdict(int))
    ewc = defaultdict(lambda: defaultdict(int))
    for week, grp in tqdm(news.groupby("week"), desc="baseline counts"):
        rows = grp["terms_f"].tolist()
        for terms in rows:
            for t in set(terms):
                twc[t][week] += 1
        if len(rows) >= MIN_WEEK_HEADLINES:
            for a, b, d in build_ppmi_graph(rows).edges(data=True):
                ewc[ek(a, b)][week] += d.get("count", 0)
    bl_weeks = sorted([w for w in week_counts if week_ts(w) <= BASELINE_END], key=week_ts)

    def bexp(c, ws):
        return float(np.mean([c.get(w, 0) for w in ws])) if ws else 0.0

    def tburst(t, w):
        return log((twc[t].get(w, 0) + ALPHA) / (bexp(twc[t], bl_weeks) + ALPHA))

    def eburst(a, b, w):
        return log((ewc[ek(a, b)].get(w, 0) + ALPHA) / (bexp(ewc[ek(a, b)], bl_weeks) + ALPHA))

    graphs, comms_by_week = {}, {}
    for week in sorted(news.loc[disc_mask & news.is_novel, "week"].unique(), key=week_ts):
        rows = news.loc[(news.week == week) & news.is_novel, "terms_f"].tolist()
        if len(rows) < MIN_NOVEL_WEEK_HEADLINES:
            continue
        g = build_ppmi_graph(rows)
        if g.number_of_nodes() >= MIN_COMMUNITY_SIZE:
            graphs[week] = g
            comms_by_week[week] = detect_comms(g)
    print(f"Novel graphs: {len(graphs)} weeks")

In [27]:
if not RUN_DETECTION:
    pass
else:
    def minmax_norm(s):
        lo, hi = s.min(), s.max()
        return pd.Series(0.5, index=s.index) if hi <= lo else (s - lo) / (hi - lo)

    next_track, prev, track_records = 0, [], []
    tbs = defaultdict(list)
    hn = news.set_index("row_id")["headline_novelty"].to_dict()

    for week in sorted(comms_by_week.keys(), key=week_ts):
        grp = news.loc[(news.week == week) & news.is_novel]
        idxs, tlists, rids = grp.index.tolist(), grp["terms_f"].tolist(), grp["row_id"].tolist()
        g = graphs[week]
        matched, week_comms = set(), []
        for c in comms_by_week[week]:
            bt, bj = None, 0.0
            for pc in prev:
                if pc["track_id"] in matched:
                    continue
                j = jaccard(c["terms"], pc["terms"])
                if j >= JACCARD_LINK and j > bj:
                    bj, bt = j, pc["track_id"]
            if bt is None:
                bt = next_track
                next_track += 1
            else:
                matched.add(bt)
            mem, mr = [], []
            for i, ts, rid in zip(idxs, tlists, rids):
                if c["terms"] & set(ts):
                    mem.append(i)
                    mr.append(rid)
            rec = {**c, "track_id": bt, "period": week, "headline_count": len(mem),
                   "headline_indices": mem, "member_row_ids": mr}
            rec["vocabulary_novelty"] = float(np.mean([1.0 if bexp(twc[t], bl_weeks) < 0.5 else 0.0 for t in rec["terms"]]))
            rec["edge_novelty"] = float(np.mean([1.0 if bexp(ewc[ek(a, b)], bl_weeks) < 0.5 else 0.0
                                                   for a, b, _ in rec["internal_edges"]])) if rec["internal_edges"] else 0.0
            rec["headline_novelty_mean"] = float(np.mean([hn.get(r, 0) for r in mr])) if mr else 0.0
            rec["top_terms"] = sorted(rec["terms"], key=lambda t: wdeg(g, t) * np.exp(tburst(t, week)), reverse=True)[:TOP_TERMS]
            rec["share"] = rec["headline_count"] / week_counts[week]
            if week_ts(week) <= BASELINE_END:
                tbs[bt].append(rec["share"])
            week_comms.append(rec)
            track_records.append(rec)
        prev = week_comms

    min_sp = 1.0 / max(week_counts.values())
    twm = defaultdict(list)
    for c in track_records:
        twm[c["track_id"]].append(c["period"])
    for c in track_records:
        bs = tbs.get(c["track_id"], [])
        c["share_growth"] = log((c["share"] + ALPHA) / ((float(np.mean(bs)) if bs else min_sp) + ALPHA))
        ws = sorted(twm[c["track_id"]], key=week_ts)
        c["first_seen"] = ws[0]

    rows = []
    for c in track_records:
        if c["headline_count"] < MIN_HEADLINES:
            continue
        nc = W_HEADLINE * c["headline_novelty_mean"] + W_EDGE * c["edge_novelty"] + W_WORD * c["vocabulary_novelty"]
        rows.append({**{k: c[k] for k in ["period", "community_id", "track_id", "headline_count", "share", "first_seen",
                                          "headline_novelty_mean", "vocabulary_novelty", "edge_novelty", "coherence", "share_growth"]},
                     "top_terms": ", ".join(c["top_terms"]), "novelty_composite": nc})
    rank_df = pd.DataFrame(rows)
    rank_df["final_score"] = 0.0
    for week, sub in rank_df.groupby("period"):
        rank_df.loc[sub.index, "final_score"] = sub["novelty_composite"] * minmax_norm(sub["coherence"]) * np.maximum(minmax_norm(sub["share_growth"].clip(lower=0)), 0.01)
    rank_df["rank_in_week"] = rank_df.groupby("period")["final_score"].rank(ascending=False, method="first").astype(int)
    rank_df.to_parquet(RANKINGS_PATH, index=False)
    print(f"Wrote {RANKINGS_PATH.name}")
    print(rank_df.sort_values(["period", "rank_in_week"]).groupby("period").head(3).to_string(index=False))

## Quick reload Part A outputs

In [24]:
from IPython.display import IFrame, display

for path in [HEADLINE_SCORES_PATH, MONTHLY_SUMMARY_PATH, TOP_TERMS_EDGES_PATH, GENAI_ADJACENT_PATH, MONTHLY_TOP_HEADLINES_PATH, WEEKLY_SUMMARY_PATH, WEEKLY_TOP_TERMS_EDGES_PATH, WEEKLY_GENAI_ADJACENT_PATH, WEEKLY_TOP_HEADLINES_PATH]:
    if not path.exists():
        print(f"missing {path.name}")
        continue
    t = pd.read_parquet(path)
    print(f"\n{path.name} ({len(t):,} rows)")
    print(t.head(5).to_string(index=False))

for path in [
    ANALYSIS_HTML_PATH,
    OUTPUT_DIR / "genai_novelty_weekly_headline.html",
    WEEKLY_DASHBOARD_HTML_PATH,
    OUTPUT_DIR / "genai_novelty_headline_dist.html",
]:
    if not path.exists():
        print(f"missing {path.name}")
        continue
    print(f"\n{path.name}")
    display(IFrame(src=str(path.resolve()), width="100%", height=520))


genai_novelty_headline_scores.parquet (298,774 rows)
                                                        Headline       date   month                  week  headline_novelty  max_baseline_sim  is_genai
Germany’s Lindner Floats Flexible Gas-Price Cap: Rheinische Post 2022-10-01 2022-10 2022-09-27/2022-10-03          0.401944          0.598056     False
  Oil Traders Regain Their Swagger With World Desperate for Fuel 2022-10-01 2022-10 2022-09-27/2022-10-03          0.253373          0.746627     False
                                           *ARK FUNDS BUY UIPATH 2022-10-01 2022-10 2022-09-27/2022-10-03          0.197983          0.802017     False
             Weaker Storm Brings Rain, Flash Floods to Carolinas 2022-10-01 2022-10 2022-09-27/2022-10-03          0.350242          0.649758     False
      Florida’s Farmers Face Widespread Destruction of Crops (1) 2022-10-01 2022-10 2022-09-27/2022-10-03          0.372861          0.627139     False

genai_novelty_monthly_summary.par


genai_novelty_weekly_headline.html



genai_novelty_headline_dist.html
